In [35]:
import os   
import joblib
import pandas as pd 
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns     
          
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder                                                        

# **PRERPOCESSING**

In [36]:
df = pd.read_csv('../data/raw/crop-recommendation.csv')
print(f"Original shape: {df.shape}")

Original shape: (2200, 23)


In [37]:
df['N_P_ratio'] = df['N'] / (df['P'] + 1e-5)
df['P_K_ratio'] = df['P'] / (df['K'] + 1e-5)
df['N_K_ratio'] = df['N'] / (df['K'] + 1e-5)

df['temp_humidity_idx'] = df['temperature'] * df['humidity']

print(f"Shape setelah feature creation: {df.shape}")

Shape setelah feature creation: (2200, 27)


In [38]:
numeric_features = df.select_dtypes(include=[np.number]).columns

def outliers_iqr(dataframe, columns):
    df_clipped = dataframe.copy()
    for col in columns:
        Q1 = df_clipped[col].quantile(0.25)
        Q3 = df_clipped[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        df_clipped[col] = np.clip(df_clipped[col], lower_bound, upper_bound)
    return df_clipped

df = outliers_iqr(df, numeric_features)

In [39]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Total Unique Labels: {len(label_mapping)}")
print(f"Mapping: {label_mapping}")

le_path = '../models/label_encoder.pkl'
os.makedirs('../models', exist_ok=True)
joblib.dump(le, le_path)
print(f"\nLabelEncoder disimpan ke {os.path.abspath(le_path)}")

Total Unique Labels: 22
Mapping: {'apple': np.int64(0), 'banana': np.int64(1), 'blackgram': np.int64(2), 'chickpea': np.int64(3), 'coconut': np.int64(4), 'coffee': np.int64(5), 'cotton': np.int64(6), 'grapes': np.int64(7), 'jute': np.int64(8), 'kidneybeans': np.int64(9), 'lentil': np.int64(10), 'maize': np.int64(11), 'mango': np.int64(12), 'mothbeans': np.int64(13), 'mungbean': np.int64(14), 'muskmelon': np.int64(15), 'orange': np.int64(16), 'papaya': np.int64(17), 'pigeonpeas': np.int64(18), 'pomegranate': np.int64(19), 'rice': np.int64(20), 'watermelon': np.int64(21)}

LabelEncoder disimpan ke /Users/faqihfirmanpratama/Documents/Ilmu Komputer/Semester 6/Daming/Project/models/label_encoder.pkl


In [40]:
X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dimensi X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Dimensi X_test: {X_test.shape}, y_test: {y_test.shape}")

Dimensi X_train: (1760, 26), y_train: (1760,)
Dimensi X_test: (440, 26), y_test: (440,)


In [41]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

In [42]:
processed_dir = '../data/processed'
models_dir    = '../models'

os.makedirs(processed_dir, exist_ok=True)

train_data = X_train_scaled.copy()
train_data['label_encoded'] = y_train.values

test_data = X_test_scaled.copy()
test_data['label_encoded'] = y_test.values

train_path  = f'{processed_dir}/train_data.csv'
test_path   = f'{processed_dir}/test_data.csv'
scaler_path = f'{models_dir}/scaler.pkl'

train_data.to_csv(train_path, index=False)
test_data.to_csv(test_path, index=False)
joblib.dump(scaler, scaler_path)

print(f"Train data  : {os.path.abspath(train_path)}")
print(f"Test data   : {os.path.abspath(test_path)}")
print(f"Scaler      :{os.path.abspath(scaler_path)}")

Train data  : /Users/faqihfirmanpratama/Documents/Ilmu Komputer/Semester 6/Daming/Project/data/processed/train_data.csv
Test data   : /Users/faqihfirmanpratama/Documents/Ilmu Komputer/Semester 6/Daming/Project/data/processed/test_data.csv
Scaler      :/Users/faqihfirmanpratama/Documents/Ilmu Komputer/Semester 6/Daming/Project/models/scaler.pkl
